In [ ]:
pip install langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.environ.get("Gemini_Api_key")

if GEMINI_API_KEY:
    print("Gemini API Key loaded successfully!")
else:
    print("WARNING: No OpenAI API Key found. Please set it in your .env file.")

## Database

In [ ]:
Orders_DB = {
    "ORD-1001": {
        "customer": "Alice Johnson", 
        "product": "laptop Pro 15", 
        "price" : 1299.99, 
        "status" : "delivered", 
        "delivery_date" : "2026-0o2-05"
    }, 
    "ORD-1002": {
        "customer": "Bob Smith", 
        "product" : "Wireless Headphones X3",
        "price" : 199.99, 
        "status" : "shipped", 
        "estimated-delivery" : 2026-2-14, 
        
    }, 
    "ORD-1003" : {
        "customer" : "Carol Davis" ,
        "product" : "Smartphone Ultra 12", 
        "price" : 899.99,
        "status": "shipped", 
        "estimated delivery" : 26-2-18,
         
    },
}

# RETURN AND REFUND POLICY 
RETURN_POLICY = {
    "standard_return_window" : "30 days from deliverey", 
    "refund_processing_time" : "5-7 business days", 
    "conditions" : [
        "Item must be in original packaging", 
        "Item must show signs of damaged caused by the costumer",
        "Proof of purchase(order ID) is required", 
    ], 
    "non_returnable" : ["Earbuds (hygine reasons)", "Software licenses"], 
}

print("BigCompany Inc. database loaded!")
print(f" - {len(Orders_DB)} orders in the system")
print(f" - Return Policy: {RETURN_POLICY['standard_return_window']}")

In [ ]:
# TOOLS 
from langchain_core.tools import tool

@tool 
def check_order_status(order_id: str) -> str: 
    """check the current status of a customer order.
    
    Use this tool when a customer askes about their order status, 
    delivery date, or order details. 
    
    Args: 
    order_id: The order ID (e.g., 'ORD-1001').
    """
    order = Orders_DB.get(order_id) 
    if not order: 
        return f"Order '{order_id}' not found. Please check the order ID and try again"
        
    info = f"Order {order_id}\n" 
    info += f" customers {order['customer']}\n"
    info += f" Product: {order['product']}\n"
    info += f" Price: ${order['price']}\n" 
    info += f"Status: {order['status']}\n"

    if order["status"] == "delivered":
        info += f" Delivered on : {order['delivery_date']}"
    else : 
            info += f" Estimated delivery : {order['estimated_delivery']}"

    return info

@tool 
def lookup_return_policy() -> str: 
    """Look up  BigCompany Inc.'s return and refund policy .
       
       use this  tool when a customer asks about returns, refunds, or the return policy.
    """
    policy = "BigCompany Inc. Return Policy:\n"
    policy += f"Return  windows: {RETURN_POLICY['standard_return_window']}\n"
    policy += f" Refund processing: {RETURN_POLICY['refund_processing_time']}\n"
    policy += f" conditions:\n"
    for conditions in RETURN_POLICY["condition"]:
        policy += f" - {condition}\n" 
    policy += " Non-returnable items:\n"
    for item in RETURN_POLICY["non_returnable"]: 
        policy += f" -{item}\n"
    return policy

@tool
def process_return_request(order_id: str, reason: str) ->str:
    """Process a return  request for a customer order. 
    
    Use this tool when a customer confirms they want to return a product.
    Only use this AFTER checking the order status and confirming the customer wants to proceed.

    Args: 
        order_id: The order ID to process the return for.
        reason: the customer's reason for returning the product.
    """

    order = Orders_DB.get(order_id)

    if not order: 
        return f"Cannot process return: Order '{order_id}'not found."

    if order["status"] != "delivered":
        return (
        f" Cannot process return: Order {order_id} has status '{order['status']}'."
        f" Returns can only be processed for delivered orders. ")
         
    return (
        f"Return request Approved  for order {order_id} ({order['product']}).\n"
        f" Reason: {reason}\n"
        f" Refund amount: ${order['price']}\n"
        f" Refund will be processed within {RETURN_POLICY['refund_processing_time']}.\n"
        f" A return shipping label has been sent  to the customer's email."
    )

# Collect all tools in a list we will give this to llm 
tools = [check_order_status, lookup_return_policy, process_return_request]

print("Tools created:") 
for t in tools:
    print(f" -{t.name}: {t.description[:60]}...")

## LLM(Large Language model)

In [ ]:
conda install -c conda-forge ipywidgets jupyterlab_widgets

In [ ]:
# Initialize Gemini Flash (FREE tier available)

# Initialize the LLM
# - model: "gpt-4o-mini" is fast and affordable, great for customer service
# - temperature: 0 means deterministic responses (no randomness)
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0
)


# Bind tools
# Give the LLM access to our tools
# After this, the LLM knows about check_order_status, lookup_return_policy, etc.
# and can decide to call them when needed.
llm_with_tools = llm.bind_tools(tools)

# The system prompt: the agent's "job description"
SYSTEM_PROMPT = """You are a friendly and helpful customer service agent for BigCompany Inc., \
an online electronics store.

Your responsibilities:
- Help customers check their order status.
- Answer questions about our return and refund policy.
- Process return requests when customers want to return a product.
- Be polite, empathetic, and professional at all times.

Important guidelines:
- Always greet the customer warmly.
- When a customer asks about an order, use the check_order_status tool.
- When a customer asks about returns or refunds, use the lookup_return_policy tool.
- Before processing a return, always confirm with the customer first.
- If you cannot help with something, let the customer know you will \
connect them with a human agent.
- Keep responses concise but helpful.
"""

print("LLM initialized with tools!")
print((f"Model: gemini-1.5-flash"))
print(f"Available tools: {[t.name for t in tools]}")


## Define the Graph State

In [ ]:
from langgraph.graph import MessagesState

# MessagesState already provides:
#   messages: Annotated[list[AnyMessage], add_messages]
#
# The add_messages reducer means:
#   - When a node returns {"messages": [new_message]},
#     the new message is APPENDED to the existing list
#   - The full conversation history is always available

# That's it! For our simple agent, MessagesState is all we need.
# No custom state class required.

print("State defined: Using MessagesState (built-in)")
print("  - Tracks: messages (full conversation history)")

## Defining the graph Nodes

In [ ]:
from langchain_core.messages import SystemMessage 
from langgraph.prebuilt import ToolNode

In [ ]:
# Node -> 1 The assistant 
# This is where the llm "thinks" about the conversation and decides what to do.

def assistant(state: MessagesState): 
    # Prepend the system prompt to give the llm its instruction 
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    #call the LLM with the full conversation + available tools 
    response = llm_with_tools.invoke(messages)

    
    # Return the LLM's response to be added to the conversations 
    return {"messages": [response]}


# Node-2: the Tool Executor 
# ToolNode automatically: 
#  - Reada tool_calls from the last AI message 
#  -Executes the requested tools
#  -Returns ToolMessage(s) with the results
tool_node = ToolNode(tools) 

print("Nodes defined : ")
print(" 1. 'assistant' - calls to llm to think and response")
print(" 2. 'tools' - executes tool calls requested by the LLM")  

## Build the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END 
from langgraph.prebuilt import tools_condition 
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
# Create a new graph with our state definition 
builder = StateGraph(MessagesState) 

# ---- Add Nodes ----  
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node) 

# -----Add Edges ----

# 1.Start ->Assistant: every conversation begins at the assistant node 
builder.add_edge(START, "assistant")

# 2. Assistant -> (conditional): after the LLM responds, check if it wants to use a tool 
#  -If the LLM's response contains tool_calls -> go to "tools" node 
#  -If not (it's a final answer) -> go to END  

builder.add_edge("tools", "assistant")

# === Compile the Graph === 
# In MemorySaver gives our agent conversation memory 
# (in production, you'd use PostgresSaver for persistence) 
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory) 

print("Graph compiled successfully")
print("\nWorkflow:") 
print(" START -> assistant ->[tools_condition] ->tools ->assistant -> ... -> END")

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Test our agent

In [ ]:
from langchain_core.messages import HumanMessage

def chat(user_message: str, thread_id: str = "Customer-session-1"):
    """send a message to the  BigCompany Inc. support agent and display the response""" 

    print(f"\n{'='*60}")
    print(f"Customer :  {user_message}")
    print(f"{'='*60}") 

# thread_id keeps track of the conversation 
# same thread_id = same conversation (agent remembers previous Messages)
config = {f"configurable": {'thread_id':thread_id}}

# stream the response so we can see each step 
for chunk in graph.stream(
    {"messages": [HumanMessage(content=user_message)]}, 
    config, 
    stream_mode="updates", 
): 
    # show which node is executing and what it produced 
    for node_name, update in chunk.items():
        last_msg = update["messages"][-1] 

        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls: 
            # the LLM is returning the tool call 
            for tc in last_msg.tool_calls:
                print(f"\n[Agent is calling tool :  {tc['name']}({tc['args']})]")

        elif node_name == "tools": 
            # Tool results are coming back 
            print(f"\n[Tool result received]")

        else: 
            # Final response  from the agent 
            print(f"\nAgent: {last_msg.content}")

print("Chat helper ready! Use chat('Your message') to talk to the agent.")

## Simple Greeting

In [15]:
chat("Hi! I need some help with my order.")


Customer :  Hi! I need some help with my order.


## Order Status

In [16]:
chat("Can you check the status of my order ORD-1002?")


Customer :  Can you check the status of my order ORD-1002?


## conversastional memory

In [18]:
chat("When will it arrive?")


Customer :  When will it arrive?


In [19]:
# Return policy question ->
chat("What is your return policy?")



Customer :  What is your return policy?


In [20]:
# processing a return (using multi-tools uMessagesState)
chat(
    "Hi, I'd like to return my Laptop Pro 15. My order ID is ORD-1001. "
    "The screen has a dead pixel.",
    thread_id="customer-session-2",
)


Customer :  Hi, I'd like to return my Laptop Pro 15. My order ID is ORD-1001. The screen has a dead pixel.


In [21]:
chat(
    "Yes, please go ahead and process the return.",
    thread_id="customer-session-2",
)


Customer :  Yes, please go ahead and process the return.


In [22]:
# Invalid order id: 
chat(
    "What's the status of order ORD-9999?",
    thread_id="customer-session-3",
)


Customer :  What's the status of order ORD-9999?


In [ ]:
## understanding what happen under the hood 

# let's inspect the state of customer-session-2 (the return conversation) 
config = {"configurable": {"thread_id": "customer-session-2"})
state = graph.get_state(configig)

print("conversation history for customer-session-2")
print(f"Total messages:{len(state.values['messages'])}\n") 

for i , msg in enumerate(state.value["messages"]):
    msg_type = type(msg).__name__ 
    content = msg.content[:100] if msg.content else "[tool call]"
    print(f"Message {i+1} ({msg_type}) : {content}")
